<a href="https://colab.research.google.com/github/OBarretoGDL/PracticasPython/blob/ForthModuleFinalProject/Projecto_Final_Modulo_4_Pokedex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Objetivos:
  * Con el lenguaje de programación Python, usando la librería requests y equipando las habilidades que has adquirido en este Módulo construirás una Pokédex obteniendo los datos de https://pokeapi.co/

  * Cuando el usuario introduzca el nombre de un Pokémon, si no existe que le mande un mensaje de error; si existe, que muestre una imagen y las estadísticas (peso, tamaño, movimientos, habilidades y tipos).

  * Posteriormente, guardarás toda la información del pokémon (junto con el link de la imagen frontal del pokémon) en un archivo .json dentro de una carpeta llamada “pokedex”.



In [ ]:
# Import Libraries

import requests #--> To consume API
import json  #--> To parse JSON
import os
import ipywidgets as widgets
from IPython.display import display, Image


In [ ]:
###### Definir funciones

##Consulta de PokeApi

# Dar formato al valor ingresado por el usuario
def is_first_char_zero(pokemon):
    if not pokemon.isnumeric():
        pokemon = pokemon.lower()
        pokemon = pokemon.lstrip().rstrip()
    elif pokemon[0] == '0':
        pokemon = pokemon[1:]
    return pokemon

# Funcion GET, obtener datos generales desde el website usando la PokeAPI y formateando datos received a tipo Json
def get_pokemon(pokemon):

  while True:
    url = f'https://pokeapi.co/api/v2/pokemon/{pokemon}'
    response = requests.get(url)
    if response.status_code == 200:
     pokemonInfo = response.json()
     return pokemonInfo
     break
    else:
        print('No se encontro el Pokemon, intente de nuevo')
    pokemon = input('Ingrese el nombre o numero del pokemon que desea buscar o escriba Salir para terminar: ')
    if pokemon.lower() == 'salir':
     break


def estructurar_informacion(resultado):
    # Extraer el nombre y el número del Pokémon
    numero = resultado['id']
    nombre = resultado['name']

    # Extraer los tipos del Pokémon
    tipos = [tipo['type']['name'] for tipo in resultado['types']]

    # Comenzar a estructurar la información
    datos = f"Numero: {numero}\nNombre: {nombre}\nPokemon Tipo: {', '.join(tipos)}\nPeso: {resultado['weight']} & Altura: {resultado['height']}\n\nHabilidades:"

    # Extraer las habilidades del Pokémon
    for elemento in resultado['abilities']:
        habilidad = elemento['ability']['name']
        datos += f"\n- {habilidad}"


    # URL del sprite (imagen) del Pokémon
    sprite_url = f"https://raw.githubusercontent.com/PokeAPI/sprites/master/sprites/pokemon/{numero}.png"
    datos += f"\n\nSprite URL: {sprite_url}"

    # Mostrar la información final
    return datos, sprite_url

def save_pokemon(pokemonInfo):
    # Extraer el nombre del Pokémon
    pokemon_name = pokemonInfo['name']

    # Crear la carpeta "pokedex" si no existe
    if not os.path.exists("pokedex"):
        os.makedirs("pokedex")

    # Definir la ruta y el nombre del archivo .json
    file_path = os.path.join("pokedex", f"{pokemon_name.lower()}.json")

    # Guardar la información del Pokémon en un archivo .json
    with open(file_path, 'w') as file:
        json.dump(pokemonInfo, file, indent=4)

    print(f"Información del Pokémon {pokemon_name} guardada en {file_path}")

def mostrar_pokemon(pokemon):
    # Obtener la información del Pokémon
    pokemon = is_first_char_zero(pokemon)
    pokemon_info = get_pokemon(pokemon)

    if pokemon_info:
        # Estructurar la información y obtener la URL del sprite
        datos, sprite_url = estructurar_informacion(pokemon_info)

        # Crear widgets para mostrar la información
        texto = widgets.Textarea(
            value=datos,
            disabled=True,
            layout=widgets.Layout(width='400px', height='200px')
         )

   # Descargar la imagen
        response = requests.get(sprite_url)

        # Crear el widget de imagen y ajustar el tamaño
        imagen = widgets.Image(
            value=response.content,
            format='png',
            layout=widgets.Layout(width='300px', height='300px')
        )

        # Mostrar los widgets
        display(texto)
        display(imagen)

        # Guardar la información del Pokémon en un archivo .json
        save_pokemon(pokemon_info)


In [ ]:
pokemon=input('Ingrese el nombre o numero del pokemon: ')
mostrar_pokemon(pokemon)
